In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA nyctaxi_project;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_demand_by_hour
COMMENT 'Viajes y tarifa promedio por día de la semana y hora de recogida'
AS
SELECT
  dayofweek(pickup_date)  AS dow_num,      -- 1 = domingo ... 7 = sábado
  pickup_dow              AS dow_name,
  pickup_hour,
  COUNT(*)                AS trips,
  ROUND(AVG(fare_amount), 2)   AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance_mi
FROM silver_trips
GROUP BY dayofweek(pickup_date), pickup_dow, pickup_hour;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_revenue_by_zone
COMMENT 'Ingresos por tarifa y volumen de viajes por ZIP de recogida'
AS
SELECT
  pickup_zip,
  COUNT(*)                     AS trips,
  ROUND(SUM(fare_amount), 2)   AS total_fare,
  ROUND(AVG(fare_amount), 2)   AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance_mi
FROM silver_trips
GROUP BY pickup_zip;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_fare_by_distance
COMMENT 'Tarifa promedio por tramo de distancia'
AS
SELECT
  CASE
    WHEN trip_distance < 1  THEN '1) 0-1 mi'
    WHEN trip_distance < 2  THEN '2) 1-2 mi'
    WHEN trip_distance < 5  THEN '3) 2-5 mi'
    WHEN trip_distance < 10 THEN '4) 5-10 mi'
    ELSE                         '5) 10+ mi'
  END AS distance_bucket,
  COUNT(*)                            AS trips,
  ROUND(AVG(fare_amount), 2)          AS avg_fare,
  ROUND(AVG(fare_amount / trip_distance), 2) AS avg_fare_per_mile,
  ROUND(AVG(duration_min), 1)         AS avg_duration_min
FROM silver_trips
GROUP BY 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_daily_kpis
COMMENT 'KPIs diarios de viajes'
AS
SELECT
  pickup_date,
  COUNT(*)                      AS trips,
  ROUND(SUM(fare_amount), 2)    AS total_fare,
  ROUND(AVG(fare_amount), 2)    AS avg_fare,
  ROUND(AVG(trip_distance), 2)  AS avg_distance_mi,
  ROUND(AVG(duration_min), 1)   AS avg_duration_min
FROM silver_trips
GROUP BY pickup_date;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_data_quality
COMMENT 'Registros rechazados por motivo, sobre el total de Bronze'
AS
WITH total AS (SELECT COUNT(*) AS n FROM bronze_trips)
SELECT
  motivo,
  COUNT(*)                               AS rejected_rows,
  ROUND(100.0 * COUNT(*) / MAX(t.n), 3)  AS pct_of_bronze
FROM (
  SELECT explode(split(reject_reasons, ', ')) AS motivo
  FROM silver_rejected_trips
) r
CROSS JOIN total t
GROUP BY motivo;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM silver_trips)                 AS silver_filas,
  (SELECT SUM(trips) FROM gold_demand_by_hour)        AS en_demand_by_hour,
  (SELECT SUM(trips) FROM gold_revenue_by_zone)       AS en_revenue_by_zone,
  (SELECT SUM(trips) FROM gold_fare_by_distance)      AS en_fare_by_distance,
  (SELECT SUM(trips) FROM gold_daily_kpis)            AS en_daily_kpis,
  (SELECT SUM(rejected_rows) FROM gold_data_quality)  AS en_data_quality;

silver_filas,en_demand_by_hour,en_revenue_by_zone,en_fare_by_distance,en_daily_kpis,en_data_quality
21792,21792,21792,21792,21792,143


In [0]:
P = "workspace.nyctaxi_project"
print("Bronze:", spark.table(f"{P}.bronze_trips").count())
print("Silver:", spark.table(f"{P}.silver_trips").count())
print("Gold daily (días):", spark.table(f"{P}.gold_daily_kpis").count())

Bronze: 21932
Silver: 21792
Gold daily (días): 60


In [0]:
from pyspark.sql import functions as F

P = "workspace.nyctaxi_project"
n_bronze = spark.table(f"{P}.bronze_trips").count()
n_silver = spark.table(f"{P}.silver_trips").count()
n_rej    = spark.table(f"{P}.silver_rejected_trips").count()
n_dias   = spark.table(f"{P}.gold_daily_kpis").count()
n_archivos = spark.table(f"{P}.bronze_trips").select("source_files").distinct().count()

print(f"Bronze: {n_bronze} | archivos: {n_archivos}")
print(f"Silver: {n_silver} | Rechazados: {n_rej} | ¿Silver+Rech = Bronze? {n_silver + n_rej == n_bronze}")
print(f"Gold daily: {n_dias} días")
print(f"Origen total: {spark.table('samples.nyctaxi.trips').count()}")

Bronze: 21932 | archivos: 60
Silver: 21792 | Rechazados: 140 | ¿Silver+Rech = Bronze? True
Gold daily: 60 días
Origen total: 21932
